# Treino e teste de mamografias CBIS-DDSM

Usa os JPGs e CSVs preparados no notebook 08. O modelo salvo é consumido pela API.

In [1]:
import sys, os
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists(): PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
from src.cnn.dataset import create_dataloaders
train_loader, val_loader, test_loader = create_dataloaders()
print('Treino:', len(train_loader.dataset), 'Validação:', len(val_loader.dataset), 'Teste:', len(test_loader.dataset))


Treino: 375 Validação: 94 Teste: 90


In [2]:
from src.cnn.train import train_cnn
# Treino com transfer learning. O primeiro uso baixa os pesos ImageNet do MobileNetV2.
# A parada antecipada interrompe o treino quando a AUC de validação deixa de melhorar.
model, metrics = train_cnn(epochs=15, finetune_epochs=8, pretrained=True, patience=5, batch_size=16, data_dir='data/images/cbis-ddsm')
metrics

C:\Users\ricoi\POSTECH\tech-challenge-fase1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026/08/07 15:56:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


MLflow não registrou o modelo PyTorch: If `serialization_format` is set to 'pt2', then input_example is required. It must be a numpy array or torch tensor, or a tuple/list of numpy arrays or torch tensors. This is because 'pt2' is a traced-graph format: PyTorch traces the model graph by virtually executing model.forward with the provided example input.


{'test_accuracy': 0.45555555555555555,
 'test_auc': 0.5298701298701299,
 'test_recall': 0.8857142857142857,
 'test_precision': 0.40789473684210525,
 'test_f1': 0.5585585585585585,
 'initial_train_loss': 0.0666978351076444,
 'initial_train_auc': 0.6972932846048424,
 'initial_val_loss': 0.07803722185657379,
 'initial_val_auc': 0.5805860805860805,
 'ft_train_loss': 0.06874076992273331,
 'ft_train_auc': 0.6677096370463079,
 'ft_val_loss': 0.07554576530101452,
 'ft_val_auc': 0.6053113553113553}

In [3]:
from pathlib import Path
from src.cnn.predict import MammoPredictor
sample = next(Path('data/images/cbis-ddsm/test/malignant').glob('*.jpg'))
MammoPredictor().predict_from_bytes(sample.read_bytes())

{'prediction': 'malignant',
 'probability_malignant': 0.6446804404258728,
 'probability_benign': 0.3553195595741272,
 'confidence': 0.6446804404258728,
 'model_used': 'mobilenet_mammo_pytorch'}